# Import libraries

In [36]:
# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pylab as plt
import seaborn as sns

# Datasets
from sklearn.datasets import load_breast_cancer

# Optimazation
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, cross_val_score

# Model
from sklearn.ensemble import RandomForestClassifier

# Metrics
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

from scipy.stats import randint

import optuna

import time

plt.style.use("ggplot")
sns.set_theme()

In [37]:
# Load datasets
data = load_breast_cancer(as_frame=True)
df= data.frame

# Features and target

In [38]:
X = df.drop("target", axis=1)
y = df['target']

# Train-test splits

In [39]:
# stratify=y (Ensures equal class proportions in train and test splits)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

# Baseline model (Random forest classifier)

In [40]:
baseline_rf = RandomForestClassifier(random_state=42)
baseline_rf.fit(X_train, y_train)
baseline_rf_predictions = baseline_rf.predict(X_test)
baseline_rf_accuracy = accuracy_score(y_test, baseline_rf_predictions)

# Cross validation

A cross-validation score is an evaluation metric that measures how well a machine learning model generalizes to unseen data. It is generated by splitting a single dataset into multiple subsets (folds), iteratively training the model on some folds while testing it on others, and averaging the resulting performance scores.

In [41]:
cross_val_score = cross_val_score(baseline_rf, X_train, y_train, cv=5, scoring="accuracy")
# cv represents no. of folds

In [42]:
print(cross_val_score)
print(cross_val_score.mean())

[0.9625 0.9875 0.9125 1.     1.    ]
0.9725000000000001


# Grid serach

In [ ]:
# Grid search
# Define a parameter grid
grid_params = {
    "n_estimators": [100, 200, 300],
    "max_depth": [5, 10, None],
    "min_samples_split":[2,5,10],
    "min_samples_leaf":[1,2,4] #  A split will only occur if both resulting
     # branch nodes have at least the number of samples defined by min_samples_leaf.
}

Grid search is a brute-force hyperparameter tuning method used in machine learning. It automatically trains and evaluates a model across an exhaustive set of predefined parameter combinations to find the configuration that yields the highest predictive accuracy.

In [ ]:
# Run grid serach
start = time.time()
# cv: The cross-validation strategy, such as cv=5, splitting your training data into 5 folds to ensure the parameters 
# generalize well.
# n_jobs: Sets the number of CPU cores to use. Setting n_jobs=-1 uses all available cores to 
# run evaluations concurrently.
grid = GridSearchCV(estimator=RandomForestClassifier(random_state=42), param_grid=grid_params, 
                    cv=5, scoring="accuracy", n_jobs=-1)
grid.fit(X_train, y_train)
grid_time = time.time() - start

In [47]:
# the best configuration
print(grid.best_params_)
print(grid.best_score_)

{'max_depth': 5, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 100}
0.9725000000000001


# Randomized Search

In [60]:
# Define parameter distributions.
random_params = {"n_estimators": randint(100,500), "max_depth":randint(3,30), "min_samples_split": randint(2,20),
                 "min_samples_leaf":randint(1,10)}

In [61]:
# Run randomized search
start = time.time()
random_search = RandomizedSearchCV(estimator=RandomForestClassifier(random_state=42), param_distributions=random_params,
                                   cv=5, n_iter=5, scoring="accuracy", n_jobs=-1)

In [62]:
random_search.fit(X_train, y_train)
random_time = time.time() - start

In [63]:
# Best results
print(random_search.best_params_)
print(random_search.best_score_)

{'max_depth': 24, 'min_samples_leaf': 1, 'min_samples_split': 10, 'n_estimators': 251}
0.9674683544303797
